In [ ]:
from importlib import reload

from occhio.distributions import SparseUniform, CorrelatedPairs
from occhio.model_grid import ModelGrid, Axis
from occhio.toy_model import ToyModel
from occhio.autoencoder import TiedLinearRelu
from occhio.visualization import (
    plot_embedding,
    plot_phase_change,
    plot_geometry,
    plot_phase_change_multi,
)
import torch
import numpy as np

In [ ]:
dist = CorrelatedPairs(6, sparsity=0.9, correlation=0.8)
dist.p_individual

In [ ]:
dag_formations = [
    np.array(
        [
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0],
        ]
    ),
    np.array(
        [
            [0.0, 1.0, 1.0],
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
        ]
    ),
    np.array(
        [
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0],
        ]
    ),
    np.array(
        [
            [0.0, 1.0, 1.0],
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0],
        ]
    ),
]

In [ ]:
device = "cpu"

N_FEATURES = 5
N_HIDDEN = 2

axis1 = Axis(label="Sparsity", values=[0.0, 0.5, 0.9])


def create_model(params):
    generator = torch.Generator(device=device).manual_seed(8)
    p_active = 1 - params["Sparsity"]
    importance = 0.9  # ["Importance"]

    return ToyModel(
        distribution=SparseUniform(N_FEATURES, p_active, generator=generator),
        importances=importance ** torch.arange(N_FEATURES, device=device),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, generator=generator),
        device=device,
    )

In [ ]:
grid = ModelGrid(
    create_model,
    axes=[axis1],
)
grid.fit(batch_size=512, n_epochs=10_000)

In [ ]:
plot_embedding(grid)

# Phase Change

In [ ]:
device = "cpu"

N_FEATURES = 3
N_HIDDEN = 2


axis1 = Axis(label="Importance", values=torch.logspace(-1, 1, 8))
axis2 = Axis(label="Density", values=torch.logspace(-2, 0, 8))


def create_model2(params):
    generator = torch.Generator(device=device).manual_seed(8)
    importance = params["Importance"]
    p_active = params["Density"]

    return ToyModel(
        distribution=SparseUniform(N_FEATURES, p_active, generator=generator),
        importances=importance ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, generator=generator),
        device=device,
    )

In [ ]:
grid = ModelGrid(
    create_model2,
    axes=[axis1, axis2],
)

In [ ]:
grid.fit(batch_size=256, n_epochs=8_000)

In [ ]:
plot_phase_change_multi(grid, up_to=3)

# Next Experiment

In [ ]:
device = "cpu"

N_FEATURES = 400
N_HIDDEN = 30


axis = Axis(label="Density", values=torch.logspace(-2, 0, 32))


def create_model(params):
    generator = torch.Generator(device=device).manual_seed(8)
    importance = 0.999  # params["Importance"]
    p_active = params["Density"]

    return ToyModel(
        distribution=SparseUniform(
            N_FEATURES, p_active, device=device, generator=generator
        ),
        importances=importance ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
    )

In [ ]:
grid = ModelGrid(
    create_model,
    axes=[axis],
)

In [ ]:
grid.fit(batch_size=256, n_epochs=10_000)

In [ ]:
plot_geometry(grid)